In [105]:
# Import libraries
import spacy
from spacy import displacy
from spacy.matcher import Matcher
import pandas as pd
nlp = spacy.load("en_core_web_sm")
import os

### Lets check our rule on a larger corpus

In [106]:
s = input()
print(s)

C:\Users\arnig\Documents\Coding_2024\python_work\UpGrad\DS_C70\Specialization_NLP\Syntactic-Processing\Syntactic-Processing_upgrad\Dependency Parsing\Dataset\active_passive.csv


In [107]:
# load the dataset csv file
data = pd.read_csv(s)
data.head()

,Active,Passive
0,He reads a novel.,A novel is read.
1,He does not cook food.,Food is not cooked by him.
2,Does he purchase books?,Are books being purchased by him?
3,They grow plants.,Plants are grown by them.
4,She teaches me.,I am taught by her.


In [108]:
# Print the shape of the dataframe.
data.shape

(40, 2)

In [109]:
# Separate out active and passive sentences in arrays.
active = data['Active']
passive = data['Passive']

### Create the rule

In [110]:
# Rule
passive_rule = [{'DEP':'nsubjpass'}]
passive_matcher = Matcher(vocab= nlp.vocab)
passive_matcher.add('Rule', [passive_rule])

In [111]:
# Test rule
print(passive_matcher(nlp(passive[0])))
print(passive_matcher(nlp(active[0])))

[(15740618714089435985, 1, 2)]
[]


In [112]:
# Checker function
def is_passive(doc, matcher):
    if len(matcher(nlp(doc)))>0:
        return True
    else:
        return False

### Check rule on active voice sentences

In [113]:
count_passive, count_active = 0, 0

for docs in zip(active, passive):
    for doc in docs:
        if not is_passive(doc, passive_matcher):
            count_active += 1
        else:
            count_passive += 1


### Check rule on passive voice sentences

In [114]:
count_passive, count_active

(38, 42)

### Let's troubleshoot

In [115]:
missed = []

for doc in passive:
    if not is_passive(doc, passive_matcher):
        missed.append(doc)

len(missed)

2

In [116]:
missed[0]

'Are books being purchased by him?'

In [117]:
missed[1]

'Is a table being bought by Ritika?'

### Let's visualize their dependency trees

In [118]:
# Normal passive sentence
displacy.render(nlp(passive[1]), style='dep')

In [119]:
# Missed sentences
for doc in missed:
    displacy.render(nlp(doc), style='dep')

In [120]:
# Passive voice sentence containing clauses
displacy.render(nlp('That she lied was suspected by everyone'), style='dep')

Some passive voice sentences containing clauses may also have 'nsubj' dependencies

In [121]:
# Active voice sentences containing clauses
displacy.render(nlp('Everyone suspected that she lied'), style='dep')

Some active voice sentences containing clauses may also have more than one 'nsubj' dependencies

[Dependencies](https://universaldependencies.org/docs/en/dep/)

### Update our rule
[Reference](https://spacy.io/usage/rule-based-matching)

In [122]:
# 1 Create list of pattern dictionaries in rule (does not work)
# passive_rule_2 = [{'DEP':'auxpass'}, {'DEP':'csubjpass'}, {'DEP':'nsubjpass'}]
# passive_matcher_2 = Matcher(vocab= nlp.vocab)
# passive_matcher_2.add('Rule', [passive_rule_2])

In [123]:
# 2 Create different rules for each pattern and pass all in a list while adding
passive_aux = [{'DEP':'auxpass'}]
passive_subj = [{'DEP':'nsubjpass'}]
passive_clause = [{'DEP':'csubjpass'}]
passive_matcher_2 = Matcher(vocab= nlp.vocab)
passive_matcher_2.add('Rule', [passive_aux, passive_subj, passive_clause])

In [124]:
# which works
count = 0
for doc in passive:
    if is_passive(doc, passive_matcher_2):
        count += 1

count

40

In [125]:
# 3 Create a single rule with all patterns using 'IN' directive
passive_rule_all = [{'DEP': {'IN': ['nsubjpass', 'csubjpass', 'auxpass']}}]
passive_matcher_3 = Matcher(vocab= nlp.vocab)
passive_matcher_3.add('Rule', [passive_rule_all])

In [126]:
# which also works
count = 0
for doc in passive:
    if is_passive(doc, passive_matcher_3):
        count += 1

count

40

## Summary
 - Always test your rules and hueristics on a larger corpus to see the effectiveness of the rules
 - One can write intricate matching rules using `matcher` object
 - Verbs are important POS in pattern matching usually forms roots of sentences and links between clauses